# Exploration des données : YouTube Shorts Prediction

Ce notebook permet d'explorer le fichier `raw_data.csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Configuration visuelle des graphiques
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Chargement des données
On va charger notre jeu de données complet regroupant les vidéos et les informations des chaînes.

In [ ]:
file_path = "raw_data.csv"
df = pd.read_csv(file_path)
df.head()

## 2. Informations Générales
Regardons la taille de notre base de données et les types de données qu'elle contient.

In [ ]:
print(f"Dimensions du dataset : {df.shape[0]} lignes et {df.shape[1]} colonnes.\n")
df.info()

## 3. Valeurs Manquantes et Doublons
Il est important de nettoyer la donnée avant de l'exploiter.

In [ ]:
missing_values = df.isnull().sum()
print("Valeurs manquantes par colonne :")
print(missing_values[missing_values > 0])

print(f"\nNombre de doublons : {df.duplicated().sum()}")

## 4. Statistiques Descriptives
Voici un aperçu des statistiques de base des colonnes au format numérique (moyenne, écart-type, min/max, quartiles).

In [ ]:
df.describe()

## 5. Visualisations
On va tracer quelques graphiques pour mieux comprendre nos données, notamment les corrélations, la distribution des vues, et la relation vues/abonnés.

In [ ]:
# Sélectionner quelques colonnes numériques pertinentes qu'on a vraisemblablement dans notre dataset
numerical_cols = [
    'views', 'likes', 'comments', 'duration', 'subscriber_count', 
    'total_videos', 'shorts_count_sample', 'avg_views', 'median_views'
]
available_num_cols = [col for col in numerical_cols if col in df.columns]

if available_num_cols:
    plt.figure(figsize=(10, 8))
    corr_matrix = df[available_num_cols].corr()
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
    plt.title('Matrice de corrélation des métriques principales')
    plt.show()

In [ ]:
if 'views' in df.columns:
    plt.figure()
    sns.histplot(df['views'], bins=50, kde=True, log_scale=True, color='blue')
    plt.title('Distribution des Vues (Échelle logarithmique)')
    plt.xlabel('Vues')
    plt.ylabel('Fréquence (Nombre de vidéos)')
    plt.show()

In [ ]:
if 'likes' in df.columns:
    plt.figure()
    sns.histplot(df['likes'], bins=50, kde=True, log_scale=True, color='green')
    plt.title('Distribution des Likes (Échelle logarithmique)')
    plt.xlabel('Likes')
    plt.ylabel('Fréquence (Nombre de vidéos)')
    plt.show()

In [ ]:
if 'subscriber_count' in df.columns and 'views' in df.columns:
    plt.figure()
    sns.scatterplot(x='subscriber_count', y='views', data=df, alpha=0.3)
    plt.xscale('log')
    plt.yscale('log')
    plt.title('Vues des vidéos vs Nombre d\'abonnés de la chaîne (Échelle log-log)')
    plt.xlabel('Nombre d\'abonnés')
    plt.ylabel('Vues de la vidéo')
    plt.show()

In [ ]:
# Vérification des anomalies d'interactions par rapport aux vues
anomalies = df[ (df['likes'] + df['comments']) > df['views'] ]
nb_anomalies = len(anomalies)

print(f"Vidéos avec plus d'interactions que de vues : {nb_anomalies} sur {len(df)} vidéos.")
if nb_anomalies > 0:
    display(anomalies[['id', 'views', 'likes', 'comments', 'virality_score']].head(10))


## 6. Histogramme du Virality Score sur clean_data
On va charger `clean_data.csv` et visualiser la distribution du score de viralité.

In [ ]:
import os
clean_data_path = '../data/clean_data.csv' if os.path.exists('../data/clean_data.csv') else 'clean_data.csv'
try:
    clean_df = pd.read_csv(clean_data_path, low_memory=False)
    if 'virality_score' in clean_df.columns:
        plt.figure()
        sns.histplot(clean_df['virality_score'], bins=50, kde=True, color='purple')
        plt.title('Distribution du Virality Score (clean_data)')
        plt.xlabel('Virality Score')
        plt.ylabel('Fréquence (Nombre de vidéos)')
        plt.show()
    else:
        print("La colonne 'virality_score' n'est pas présente dans clean_data.csv")
except FileNotFoundError:
    print("Fichier clean_data.csv introuvable.")
